In [1]:
import os
from pathlib import Path
import pymupdf4llm

import re

from collections import Counter

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


In [2]:
PDF_DIR = Path("./raw_pdfs")
MARKDOWN_DIR = Path("./parsed_markdowns")
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

> # IMPORTANT Conversion exception
> 
> `Bowel_Cancer_UK_Colonic_Stenting_V2.1.pdf` has a faulty structure and cannot be parsed therefore was manually parsed, please download it from OneDrive.
> This feature might be added in the future using Image OCR. 

In [3]:
def convert_pdf_to_md(pdf_path: Path, output_dir: Path):
    try:
        if("raw_pdfs/Bowel_Cancer_UK_Colonic_Stenting_V2.1.pdf" in str(pdf_path)):
            raise Exception("Bowel_Cancer_UK_Colonic_Stenting_V2.1.pdf has a faulty structure and was manually parsed, please download it from OneDrive")
        markdown = pymupdf4llm.to_markdown(str(pdf_path))
        output_file = output_dir / (pdf_path.stem + ".md")
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(markdown)
        print(f"[OK] {pdf_path.name} -> {output_file.name}")
    except Exception as e:
        print(f"[FAIL] {pdf_path.name}: {e}")

In [4]:
pdf_files = list(PDF_DIR.glob("*.pdf"))

if not pdf_files:
    print("No PDF files found in raw_pdfs/")

for pdf in pdf_files:
    convert_pdf_to_md(pdf, MARKDOWN_DIR)

[FAIL] Bowel_Cancer_UK_Colonic_Stenting_V2.1.pdf: Bowel_Cancer_UK_Colonic_Stenting_V2.1.pdf has a faulty structure and was manually parsed, please download it from OneDrive
[OK] WOCN_Catheterization_Of_Urinary_Stoma_2018.pdf -> WOCN_Catheterization_Of_Urinary_Stoma_2018.md
[OK] NCPD_Nursing_Care_For_Patients_After_Urostomy_Surgery.pdf -> NCPD_Nursing_Care_For_Patients_After_Urostomy_Surgery.md
[OK] Bowel_Cancer_UK_Practical_Tips_Booklet_2025.pdf -> Bowel_Cancer_UK_Practical_Tips_Booklet_2025.md
[OK] Bowel_Cancer_UK_The_Facts_About_Bowel_Cancer_Easy_Read_Booklet.pdf -> Bowel_Cancer_UK_The_Facts_About_Bowel_Cancer_Easy_Read_Booklet.md
[OK] Bowel_Cancer_UK_About_Stoma_Reversal_2024.pdf -> Bowel_Cancer_UK_About_Stoma_Reversal_2024.md
[OK] NCCN_Colon_Cancer_Guideline_For_Patient_2025.pdf -> NCCN_Colon_Cancer_Guideline_For_Patient_2025.md
[OK] Bowel_Cancer_UK_Eating_Well.pdf -> Bowel_Cancer_UK_Eating_Well.md
[OK] ACS_Colorectal_Cancer_Fact_Sheet.pdf -> ACS_Colorectal_Cancer_Fact_Sheet.md


# Clean clean clean (preserve structure)


In [5]:
CLEANED_MARKDOWN_DIR = Path("./cleaned_markdowns")
CLEANED_MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
def load_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

In [7]:
HEADING_RE = re.compile(r'^\s*#{1,6}\s+')          # Markdown headings
LIST_RE = re.compile(r'^\s*([*\-]|\d+\.)\s+')     # bullets or numbered lists
BLANK_RE = re.compile(r'^\s*$')
DIVIDER_RE = re.compile(r'\s*---\s*')

TABLE_ROW_RE = re.compile(r'^\s*\|.*\|\s*$')
TABLE_SEPARATOR_RE = re.compile(r'^\s*\|?\s*:?-{2,}:?\s*(\|\s*:?-{2,}:?\s*)+\|?\s*$')

SENTENCE_END_RE = re.compile(r'[.!?:]\s*$')
LOWER_START_RE = re.compile(r'^[a-z(]')


In [8]:
def remove_headers_footers(text):
    lines = text.split("\n")
    
    cleaned = []
    for line in lines:
        # Page numberings
        if re.search(r'^(\*\*)?\d+\b(\*\*)?(?![\.\)-])', line):
            continue
        if len(line.strip()) < 3:
            cleaned.append(line)
            continue
        if line.isupper() and len(line.split()) > 5:
            # heuristic: large uppercase banner headers
            continue
        cleaned.append(line)
    
    return "\n".join(cleaned)

In [9]:
def is_protected(line):
    return (
        BLANK_RE.match(line) or
        LIST_RE.match(line) or
        TABLE_ROW_RE.match(line) or
        TABLE_SEPARATOR_RE.match(line)
    )

def detect_repeating_lines(text, min_length=5, top_k=20, freq_threshold=0.02):
    raw_lines = text.split("\n")
    
    # normalize but keep original for filtering
    lines = [l.strip() for l in raw_lines]
    
    # filter trivial + protected lines
    filtered = [
        l for l in lines
        if len(l) >= min_length and not is_protected(l)
    ]
    
    total_lines = len(filtered)
    counter = Counter(filtered)
    
    repeating = set()
    
    for line, count in counter.items():
        freq = count / total_lines if total_lines > 0 else 0
        
        if freq >= freq_threshold and count > 2:
            repeating.add(line)
    
    return repeating

In [10]:
def remove_repeating_lines(text):
    lines = text.split("\n")
    
    repeating_lines = detect_repeating_lines(text)
    
    cleaned = []
    for line in lines:
        stripped = line.strip()
        
        # remove statistically repeated lines
        if stripped in repeating_lines:
            continue
            
        cleaned.append(line)
    
    return "\n".join(cleaned)    

In [11]:
def normalize_whitespace(text):
    # remove starting spaces
    text = re.sub(r'^[ \t]+', '', text, flags=re.MULTILINE)
    
    # remove trailing spaces
    text = re.sub(r'[ \t]+$', '', text, flags=re.MULTILINE)
    
    # normalize multiple spaces
    text = re.sub(r'[ ]{2,}', ' ', text)
    
    # normalize excessive newlines (keep paragraph breaks)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text

In [12]:
def remove_bold_asterisks(text):
    # Remove **bold** markers but keep inner text
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    
    return text

In [13]:
def fix_broken_words(text):
    # join hyphenated line breaks: exam-\nple → example
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    text = re.sub(r'([a-zA-Z0-9]+)\s+(-\w+)', r'\1\2', text)
    
    return text

In [14]:
def normalize_bullets(text):
    # Replace common bullet characters with "- "
    text = re.sub(r'^\s*#*\s*([•▪◦●○]|(`o`))+\s*', '\n- ', text, flags=re.MULTILINE)
    
    return text

In [15]:
def is_table_line(line):
    return TABLE_ROW_RE.match(line) or TABLE_SEPARATOR_RE.match(line)

In [16]:
def merge_lines(text):
    lines = text.split("\n")
    merged = []
    in_table = False
    i = 0

    while i < len(lines):
        line = lines[i]
        stripped = line.strip()

        # --- TABLE HANDLING ---
        if is_table_line(line):
            merged.append(line)
            in_table = True
            i += 1
            continue
        else:
            if in_table:
                in_table = False

        # Divider lines (REMOVE)
        if DIVIDER_RE.match(line):
            i += 1
            continue

        # --- MODIFIED BLANK LINE HANDLING ---
        if BLANK_RE.match(line):
            if merged and i + 1 < len(lines):
                prev = merged[-1]
                nxt = lines[i + 1].strip()

                # soft break: mid-sentence continuation
                if (
                    not SENTENCE_END_RE.search(prev)
                    and LOWER_START_RE.match(nxt)
                ):
                    i += 1
                    continue  # skip blank line

            merged.append("")
            i += 1
            continue

        # Headings → always new block
        if HEADING_RE.match(line):
            merged.append(stripped)
            i += 1
            continue

        # New list item → always new block
        if LIST_RE.match(line):
            merged.append(stripped)
            i += 1
            continue

        # Continuation logic
        if merged:
            prev = merged[-1]

            # do not merge into table
            if is_table_line(prev):
                merged.append(stripped)
                i += 1
                continue

            # previous is list item → append continuation
            if LIST_RE.match(prev):
                merged[-1] = prev + " " + stripped
                i += 1
                continue

            # previous is normal paragraph → merge ONLY if not sentence end
            if not HEADING_RE.match(prev) and not SENTENCE_END_RE.search(prev) and LOWER_START_RE.match(stripped):
                merged[-1] = prev + " " + stripped
                i += 1
                continue

        # fallback
        merged.append(stripped)
        i += 1

    return "\n".join(merged)

In [19]:
def clean_text(text):
    text = normalize_whitespace(text)
    text = remove_bold_asterisks(text)
    text = fix_broken_words(text)
    text = normalize_bullets(text)
    text = remove_repeating_lines(text)
    text = merge_lines(text)
    text = remove_headers_footers(text)
    text = normalize_whitespace(text)
    return text

In [20]:
for file_path in MARKDOWN_DIR.glob("*.md"):
    raw = load_file(file_path)
    cleaned = clean_text(raw)
    
    output_path = CLEANED_MARKDOWN_DIR / (file_path.stem + ".md")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(cleaned)